# Vecteurs DreaMS

DreaMS (Bushuiev *et al.*, *Nature Biotechnology*, 2025) est un transformer pré-entraîné sur des millions de spectres MS/MS. Ce notebook calcule son vecteur de 1 024 nombres pour chacun des spectres exportés par `casmi_final.ipynb` (section 8). Il ne fait que ça : `casmi_final.ipynb` se sert ensuite de ces vecteurs pour les modèles « DreaMS + kNN » et « DreaMS + tête ».

DreaMS a besoin d'un GPU NVIDIA pour aller vite. Deux façons de l'exécuter :

**Sur Google Colab (GPU gratuit), ce que nous avons fait :**
1. Exécuter `casmi_final.ipynb` une première fois : sa section 8 écrit `data/final/dreams/spectres_dreams.parquet` (443 Mo).
2. Déposer ce fichier dans un dossier `CASMI` de votre Google Drive. Si vous choisissez un autre dossier, adaptez `DRIVE_DIR` plus bas.
3. Ouvrir ce notebook dans Colab, choisir **Exécution > Modifier le type d'exécution > GPU T4**, puis **Tout exécuter**, en gardant l'onglet ouvert. Comptez de 30 à 60 minutes, installation comprise ; la cellule de mesure donne une estimation.
4. Télécharger `dreams_embeddings.npy` (486 Mo) depuis le Drive et le copier dans `data/final/dreams/`.

**Sur un PC :** exécuter ce notebook depuis le même dossier que `casmi_final.ipynb`, après la première exécution de celui-ci. Il lit et écrit directement dans `data/final/dreams/`. Il faut `git`, une connexion Internet, et environ 4 Go de disque pour l'environnement de DreaMS et ses poids, installés dans `.dreams/`. Sans GPU NVIDIA, DreaMS tourne sur le processeur, mais il faut alors compter plusieurs heures ; la cellule de mesure donne une estimation avant le calcul complet.

Une fois les vecteurs en place, relancer `casmi_final.ipynb` : les modèles déjà calculés sont relus sur le disque, et seuls les modèles DreaMS sont entraînés.

In [ ]:
import os
import shutil
import subprocess
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd

try:
    import google.colab  # noqa: F401  (présent seulement sur Colab)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

TOOLS_DIR = Path("/content") if IN_COLAB else Path(".dreams").resolve()  # dépôt DreaMS, environnement, fichiers temporaires
TOOLS_DIR.mkdir(exist_ok=True)
HAS_GPU = shutil.which("nvidia-smi") is not None
if HAS_GPU:
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True)
    print("GPU :", gpu.stdout.strip())
else:
    print("Aucun GPU NVIDIA : DreaMS tournera sur le processeur, en plusieurs heures."
          + (" Sur Colab : Exécution > Modifier le type d'exécution > GPU T4." if IN_COLAB else ""))
print("Exécution sur", "Google Colab" if IN_COLAB else "un PC", "|", os.cpu_count(), "processeurs")

### Le fichier de spectres

Sur Colab, le fichier est lu sur le Drive : Colab demande l'autorisation d'y accéder, et il faut cocher toutes les cases. Si le Drive refuse (« credential propagation was unsuccessful »), la cellule cherche le fichier déposé à la main dans `/content`, par le panneau Fichiers à gauche. Sur un PC, le fichier est lu dans `data/final/dreams/`. À regarder : le nombre de spectres, qui doit être celui qu'affiche `casmi_final.ipynb` (237 372 pour nous).

In [ ]:
if IN_COLAB:
    DRIVE_DIR = Path("/content/drive/MyDrive/CASMI")  # dossier du Drive qui contient spectres_dreams.parquet
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        WORK = DRIVE_DIR
    except Exception as error:  # Drive refusé : fichier déposé à la main dans /content
        print(f"Drive non monté ({error}).")
        print("Glissez spectres_dreams.parquet dans le panneau Fichiers (à gauche), dossier /content.")
        WORK = Path("/content")
else:
    WORK = Path("data/final/dreams")  # écrit par casmi_final.ipynb, section 8
SPECTRA = WORK / "spectres_dreams.parquet"
OUT = WORK / "dreams_embeddings.npy"
assert SPECTRA.exists(), f"Fichier introuvable : {SPECTRA}. Exécutez d'abord casmi_final.ipynb (section 8)."

spectra = pd.read_parquet(SPECTRA)
n_peaks = spectra["mzs"].map(len)
print(f"{len(spectra):,} spectres, {n_peaks.sum():,} pics ; spectres sans pic : {(n_peaks == 0).sum()}")

### Installer DreaMS

DreaMS impose des versions anciennes (Python 3.11, torch 2.2.1, numpy 1.25), qui ne s'installent pas à côté de celles du projet. On lui crée donc un environnement à part avec `uv`, qui télécharge lui-même Python 3.11. Comptez environ 5 minutes. À regarder : la dernière ligne indique la version de torch, et `CUDA : True` si le GPU est utilisable.

In [ ]:
REPO_DIR, ENV_DIR = TOOLS_DIR / "DreaMS", TOOLS_DIR / "dreams_env"
ENV_PYTHON = ENV_DIR / ("Scripts/python.exe" if os.name == "nt" else "bin/python")
UV = [sys.executable, "-m", "uv"]

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "-q", "https://github.com/pluskal-lab/DreaMS.git", str(REPO_DIR)], check=True)
if not ENV_DIR.exists():
    subprocess.run([*UV, "venv", "-q", "-p", "3.11", str(ENV_DIR)], check=True)
install = subprocess.run([*UV, "pip", "install", "-q", "--python", str(ENV_PYTHON), "-e", str(REPO_DIR)],
                         capture_output=True, text=True)
assert install.returncode == 0, "Installation de DreaMS échouée :\n" + install.stderr[-3000:]
check = subprocess.run([str(ENV_PYTHON), "-c", "import torch, dreams; print('torch', torch.__version__, '| CUDA :', torch.cuda.is_available())"],
                       capture_output=True, text=True)
print(check.stdout or check.stderr[-3000:])

### Écrire les spectres au format MGF

DreaMS lit un fichier texte `.mgf`. Pour chaque spectre, un bloc `BEGIN IONS` … `END IONS` donne le précurseur (`PEPMASS`), la charge, puis une ligne « m/z intensité » par pic. Les blocs sont écrits dans l'ordre du fichier de spectres : la ligne i des vecteurs correspondra au spectre i. Un spectre sans pic recevrait un pic fictif au précurseur, pour que DreaMS ne le saute pas ; la cellule « Le fichier de spectres » dit s'il y en a.

Le calcul se fait sans suivre les gradients (`torch.no_grad`) : sans cela, torch garde en mémoire de quoi entraîner le modèle, et la mémoire du GPU sature dès le premier paquet de spectres.

In [ ]:
def write_mgf(df, path):
    """Écrit les spectres de df au format MGF, un bloc par spectre, dans l'ordre des lignes."""
    with open(path, "w") as f:
        for prec, charge, mzs, intensities in zip(df["precursor_mz"], df["charge"], df["mzs"], df["intensities"]):
            if len(mzs) == 0:  # spectre vide : un pic fictif au précurseur
                mzs, intensities = [prec], [1.0]
            f.write(f"BEGIN IONS\nPEPMASS={prec:.6f}\nCHARGE={charge}\n")
            f.write("".join(f"{m:.5f} {i:.6g}\n" for m, i in zip(mzs, intensities)))
            f.write("END IONS\n\n")

EMBED_SCRIPT = TOOLS_DIR / "embed.py"
EMBED_SCRIPT.write_text("""
import sys
import numpy as np
import torch
from dreams.api import dreams_embeddings
with torch.no_grad():  # prédiction seule : sans cela, torch garde de quoi calculer des gradients et sature le GPU
    embs = dreams_embeddings(sys.argv[1], batch_size=int(sys.argv[3]), progress_bar=False)
np.save(sys.argv[2], embs.astype(np.float16))
""")
BATCH_SIZE = 128 if HAS_GPU else 32  # spectres par paquet ; descendre à 32 si la mémoire du GPU sature

def embed(mgf_path, npy_path):
    """Vecteurs DreaMS des spectres d'un fichier MGF, calculés dans l'environnement DreaMS ; renvoie la durée en secondes."""
    start = time.time()
    run = subprocess.run([str(ENV_PYTHON), str(EMBED_SCRIPT), str(mgf_path), str(npy_path), str(BATCH_SIZE)],
                         capture_output=True, text=True)
    assert run.returncode == 0, "DreaMS a échoué :\n" + run.stderr[-3000:]
    return time.time() - start

### Mesurer la vitesse

On calcule d'abord les vecteurs d'un petit lot de spectres (2 000 avec un GPU, 500 sans). Le premier appel télécharge aussi les poids du modèle (environ 1,2 Go, depuis Hugging Face) : on le lance donc deux fois, et on chronomètre le second. À regarder : la forme des vecteurs (n × 1 024) et l'estimation de durée pour tous les spectres.

In [ ]:
N_TRIAL = 2000 if HAS_GPU else 500
trial_mgf, trial_npy = TOOLS_DIR / "essai.mgf", TOOLS_DIR / "essai.npy"
write_mgf(spectra.head(N_TRIAL), trial_mgf)
embed(trial_mgf, trial_npy)            # télécharge les poids
seconds = embed(trial_mgf, trial_npy)  # chronométré
trial = np.load(trial_npy)
speed = len(trial) / seconds
print(f"Vecteurs : {trial.shape} ; {speed:.0f} spectres par seconde, "
      f"soit environ {len(spectra) / speed / 60:.0f} min pour les {len(spectra):,} spectres")

### Tous les spectres

Le calcul complet. Les vecteurs sont enregistrés en float16, qui prend deux fois moins de place et suffit pour une similarité cosinus. À regarder : autant de vecteurs que de spectres.

In [ ]:
all_mgf, all_npy = TOOLS_DIR / "spectres.mgf", TOOLS_DIR / "dreams_embeddings.npy"
write_mgf(spectra, all_mgf)
seconds = embed(all_mgf, all_npy)
embeddings = np.load(all_npy)
assert embeddings.shape[0] == len(spectra), f"{embeddings.shape[0]} vecteurs pour {len(spectra)} spectres"
assert np.isfinite(embeddings).all(), "Vecteurs non finis"
np.save(OUT, embeddings)
print(f"{embeddings.shape[0]:,} vecteurs de {embeddings.shape[1]} nombres en {seconds / 60:.0f} min, enregistrés dans {OUT}")

### Et ensuite

- **Sur Colab :** télécharger `dreams_embeddings.npy` depuis le Drive (ou, sans Drive, depuis le panneau Fichiers : clic droit, Télécharger) et le copier dans `data/final/dreams/`, à côté de `spectres_dreams.parquet`.
- **Sur un PC :** rien à faire, le fichier est déjà dans `data/final/dreams/`.

Relancer ensuite `casmi_final.ipynb` : sa section 8 doit afficher le nombre de vecteurs, et les modèles DreaMS entrent dans le criblage.